# Backfill – Car Workshop

Generates historical facts by running `daily.ipynb` once per date
(`dbutils.notebook.run`, parallel). Each date owns a disjoint ID block and a
deterministic seed, so runs are reproducible and IDs never collide.

| widget | meaning |
|---|---|
| `START_DATE` / `END_DATE` | date range, END blank = yesterday |
| `SCALE_FACTOR` | passed through to `daily.ipynb` (blank = 1.0 ≈ 100K rows/day) |
| `PARALLELISM` | concurrent notebook runs on this cluster (default 4) |
| `SKIP_EXISTING` | skip dates already generated (safe to re-run after failures) |

Completion marker: `fact_employee_schedules` files – `daily.ipynb` writes that table
**last**, so its file for a date means the whole day finished.

Volume estimate: `days × 100K × SCALE_FACTOR` rows. Full history 2020-01-01 → today
at `SCALE_FACTOR = 10` is ~2.4B rows – start smaller.

In [0]:
from datetime import date, timedelta

dbutils.widgets.text('START_DATE', '2026-01-01', 'Start date (YYYY-MM-DD)')
dbutils.widgets.text('END_DATE', '', 'End date (YYYY-MM-DD, blank = yesterday)')
dbutils.widgets.text('SCALE_FACTOR', '', 'Scale factor (blank = 1.0)')
dbutils.widgets.text('PARALLELISM', '4', 'Concurrent notebook runs')
dbutils.widgets.dropdown('SKIP_EXISTING', 'true', ['true', 'false'], 'Skip already generated dates')

START_DATE = date.fromisoformat(dbutils.widgets.get('START_DATE').strip())
_raw_end = dbutils.widgets.get('END_DATE').strip()
END_DATE = date.fromisoformat(_raw_end) if _raw_end else date.today() - timedelta(days=1)
SCALE_FACTOR = dbutils.widgets.get('SCALE_FACTOR').strip()  # passed through to daily.ipynb
PARALLELISM = int(dbutils.widgets.get('PARALLELISM'))
SKIP_EXISTING = dbutils.widgets.get('SKIP_EXISTING') == 'true'

assert START_DATE <= END_DATE, 'START_DATE must be <= END_DATE'
dates = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)]

est_rows = int(len(dates) * 100_000 * float(SCALE_FACTOR or 1.0))
print(f'{len(dates)} days: {START_DATE} .. {END_DATE}')
print(f'estimated volume: ~{est_rows:,} rows (SCALE_FACTOR = {SCALE_FACTOR or "1.0"})')

## Skip dates that are already generated

In [0]:
FACT_OUTPUT_DIR = '/Volumes/car_workshop/fact/fact_files'
MARKER_TABLE = 'fact_employee_schedules'  # written last by daily.ipynb -> completion marker

done = set()
if SKIP_EXISTING:
    try:
        for f in dbutils.fs.ls(f'{FACT_OUTPUT_DIR}/{MARKER_TABLE}'):
            # fact_employee_schedules_20260705_ab12cd34.parquet
            token = f.name.replace('.parquet', '').split('_')[-2]
            done.add(date(int(token[:4]), int(token[4:6]), int(token[6:8])))
    except Exception:
        pass  # directory does not exist yet -> nothing generated so far

todo = [d for d in dates if d not in done]
print(f'already generated: {len(dates) - len(todo):,}, to run: {len(todo):,}')

## Run daily.ipynb per date (parallel)

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor


def run_day(target_date):
    t0 = time.time()
    try:
        dbutils.notebook.run('./daily', 3600, {
            'TARGET_DATE': target_date.isoformat(),
            'SCALE_FACTOR': SCALE_FACTOR,
        })
        return {'date': target_date, 'status': 'OK', 'seconds': round(time.time() - t0, 1), 'error': ''}
    except Exception as e:
        return {'date': target_date, 'status': 'FAILED', 'seconds': round(time.time() - t0, 1),
                'error': str(e)[:200]}


t0 = time.time()
results = []
with ThreadPoolExecutor(max_workers=PARALLELISM) as pool:
    for i, res in enumerate(pool.map(run_day, todo), 1):
        results.append(res)
        print(f"[{i}/{len(todo)}] {res['date']} {res['status']} ({res['seconds']}s)")

print(f'backfill finished in {(time.time() - t0) / 60:.1f} min')

## Summary

In [0]:
import pandas as pd

if results:
    summary = pd.DataFrame(results)
    print(summary['status'].value_counts().to_string())
    failed = summary[summary['status'] == 'FAILED']
    if len(failed):
        print()
        print('FAILED dates - just re-run this notebook with SKIP_EXISTING=true,')
        print('only the missing dates will be retried:')
        print(failed.to_string(index=False))
else:
    print('nothing to do - all dates already generated')

## Ingest everything

One `availableNow` pass of Auto Loader picks up all files generated above.

In [0]:
%run ./autoloader